# End-to-End ETL Pipeline (Local)

Runs the full WeatherSpeak PH pipeline locally using `modal_etl/core/` modules.

**Pipeline:** PDF → OCR markdown + metadata → radio scripts + TTS text → MP3

The same `run_step1 / run_step2 / run_step3` functions are used by the Modal ETL in production.
Output mirrors the Modal Volume layout: `output/{stem}/ocr.md`, `radio_{lang}.md`, etc.

Set `force=True` in any step cell to re-run that step even if outputs already exist.

In [1]:
import sys
from pathlib import Path

# Make modal_etl importable from notebook directory
sys.path.insert(0, str(Path.cwd().parent))

from modal_etl.core.ocr import run_step1
from modal_etl.core.scripts import run_step2
from modal_etl.core.tts import run_step3

In [2]:
# ── Configuration ────────────────────────────────────────────────────────
STEM          = "PAGASA_25-TC22_Verbena_TCB#24"
PDF_PATH      = Path("../data/bulletin-archive/archive/pagasa-25-TC22") / f"{STEM}.pdf"
OUTPUT_DIR    = Path("10-etl-e2e/output")
OLLAMA_URL    = "http://localhost:11434"
LANGUAGES     = ["ceb", "tl", "en"]
TTS_MODELS_DIR = Path.home() / ".cache" / "huggingface" / "hub"
FORCE_RUN       = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"PDF:        {PDF_PATH}  (exists={PDF_PATH.exists()})")
print(f"Output dir: {OUTPUT_DIR.resolve()}")
print(f"Ollama URL: {OLLAMA_URL}")

PDF:        ../data/bulletin-archive/archive/pagasa-25-TC22/PAGASA_25-TC22_Verbena_TCB#24.pdf  (exists=True)
Output dir: /Users/josereyes/Dev/gemma4-hackathon/notebooks/10-etl-e2e/output
Ollama URL: http://localhost:11434


## Step 1: OCR — PDF → Markdown + Metadata

Sends each PDF page to Gemma 4 E4B via Ollama vision API.
Writes `ocr.md`, `chart.png`, and `metadata.json` to `output/{stem}/`.

In [3]:
stem_dir = run_step1(PDF_PATH, OUTPUT_DIR, ollama_url=OLLAMA_URL, force=FORCE_RUN)
print(f"\nStep 1 complete → {stem_dir}")

# Preview OCR markdown
ocr_md = (stem_dir / "ocr.md").read_text(encoding="utf-8")
print(f"\n--- ocr.md preview (first 500 chars) ---")
print(ocr_md[:500])

# Pretty-print metadata
import json
metadata = json.loads((stem_dir / "metadata.json").read_text(encoding="utf-8"))
print(f"\n--- metadata.json ---")
print(json.dumps(metadata, indent=2, ensure_ascii=False)[:1000])

[run_step1] PAGASA_25-TC22_Verbena_TCB#24: wrote ocr.md (8605 chars)
[run_step1] PAGASA_25-TC22_Verbena_TCB#24: saved chart.png (page 1)
[run_step1] PAGASA_25-TC22_Verbena_TCB#24: wrote metadata.json

Step 1 complete → 10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24

--- ocr.md preview (first 500 chars) ---
<!-- Page 1 -->

TROPICAL CYCLONE BULLETIN NO. 24
VERBENA
(formerly known as PEARL)

"VERBENA" WEAKENS WHILE MOVING WEST SOUTHWESTWARD SLOWLY

General Remarks:
As of 0000H Tuesday, 24 February, VERBENA is located at 11.2°N 116.8°E. The storm is currently at its weakest, with maximum sustained winds of 35 kph. It is moving generally West-Southwestward.

Forecast Track:

| Date and Time | Location (Center) | Latitude | Longitude | Min. Wind (kph) | Movement (Stationary) |
| :---: | :---: | :---: |

--- metadata.json ---
{
  "bulletin_type": "TCA",
  "storm": {
    "name": "LAKBA",
    "category": "Tropical Storm",
    "former_name": null,
    "wind_signal": null
  },
  "issuance": {
 

## Step 2: Radio Scripts + TTS Text

Generates a spoken weather announcement and TTS-optimised plain text for each language.
Writes `radio_{lang}.md` and `tts_{lang}.txt` to `output/{stem}/`.

In [4]:
for lang in LANGUAGES:
    radio_path = run_step2(STEM, lang, OUTPUT_DIR, ollama_url=OLLAMA_URL, force=FORCE_RUN)
    print(f"\n{'='*60}")
    print(f"[{lang.upper()}] {radio_path.name}")
    print('='*60)
    print(radio_path.read_text(encoding="utf-8")[:400])
print("\nStep 2 complete.")

[run_step2] PAGASA_25-TC22_Verbena_TCB#24/ceb: using hybrid input (metadata + OCR)
[run_step2] PAGASA_25-TC22_Verbena_TCB#24/ceb: wrote radio + tts files

[CEB] radio_ceb.md
Karon, kusog nga bagyo LAKBA. Naglihok kini pagpaayo paingon sa hilaga-kasadlaw-kasadlaw paingon sa kalibutan sa Pilipinas.

Adunay mga pasidaan sa pag-ulan nga grabe hangtod sa Bicol, Samar, Eastern Visayas, ug bahin sa Southern Luzon.

Walay signal nga gi-isaad karon, apan pag-andam pa gihapon.

Ang kinahanglan ninyong buhaton: pag-andam sa pag-ulan ug pagpasaka sa pag-amping. Likayi ang pagduol
[run_step2] PAGASA_25-TC22_Verbena_TCB#24/tl: using hybrid input (metadata + OCR)
[run_step2] PAGASA_25-TC22_Verbena_TCB#24/tl: wrote radio + tts files

[TL] radio_tl.md
Mangyaring makinig nang mabuti. Ang bagyo ay si LAKBA. Isa itong tropical cyclone na unti-unting humihina.

Kasalukuyan itong gumagalaw sa pangkalahatang direksyong hilaga-hilagang-kanluran patungo sa mga lugar sa Pilipinas.

May mga babala sa pag-ulan ng

## Step 3: TTS Synthesis → MP3

Synthesizes MP3 audio for each language using:
- English: Coqui XTTS v2 (speaker: Damien Black)
- Tagalog / Cebuano: Facebook MMS VITS

Writes `audio_{lang}.mp3` to `output/{stem}/`.

In [5]:
for lang in LANGUAGES:
    mp3_path = run_step3(STEM, lang, OUTPUT_DIR, TTS_MODELS_DIR, force=FORCE_RUN)
    size_kb = mp3_path.stat().st_size // 1024
    print(f"[{lang.upper()}] {mp3_path.name}  ({size_kb} KB)")
print("\nStep 3 complete.")

/Users/josereyes/Dev/gemma4-hackathon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[MMSSynthesizer] loading facebook/mms-tts-ceb on cpu
[run_step3] PAGASA_25-TC22_Verbena_TCB#24/ceb: wrote 10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24/audio_ceb.mp3
[CEB] audio_ceb.mp3  (759 KB)
[MMSSynthesizer] loading facebook/mms-tts-tgl on cpu
[run_step3] PAGASA_25-TC22_Verbena_TCB#24/tl: wrote 10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24/audio_tl.mp3
[TL] audio_tl.mp3  (775 KB)
[CoquiXTTSSynthesizer] loading on cpu
[run_step3] PAGASA_25-TC22_Verbena_TCB#24/en: wrote 10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24/audio_en.mp3
[EN] audio_en.mp3  (537 KB)

Step 3 complete.


In [6]:
from IPython.display import Audio, display, Markdown

for lang in LANGUAGES:
    mp3_path = OUTPUT_DIR / STEM / f"audio_{lang}.mp3"
    if mp3_path.exists():
        display(Markdown(f"**{lang.upper()}**"))
        display(Audio(str(mp3_path)))

**CEB**

**TL**

**EN**

## Output Summary

In [7]:
stem_dir = OUTPUT_DIR / STEM
print(f"Artefacts in {stem_dir.resolve()}:\n")
for f in sorted(stem_dir.iterdir()):
    size = f.stat().st_size
    unit = 'KB' if size >= 1024 else 'B'
    val = size // 1024 if size >= 1024 else size
    print(f"  {f.name:<35}  {val:>6} {unit}")

Artefacts in /Users/josereyes/Dev/gemma4-hackathon/notebooks/10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24:

  audio_ceb.mp3                           759 KB
  audio_en.mp3                            537 KB
  audio_tl.mp3                            775 KB
  chart.png                               557 KB
  metadata.json                             1 KB
  ocr.md                                    8 KB
  radio_ceb.md                            589 B
  radio_en.md                             459 B
  radio_tl.md                             698 B
  tts_ceb.txt                             621 B
  tts_en.txt                              425 B
  tts_tl.txt                              720 B
